# EDA of results of SF Model v1

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb
import numpy as np

In [37]:
df = pd.read_csv("./results.csv")
df_data = pd.read_csv("./data/data.csv")

In [58]:
# Anomalous data as per the model
anomalies = df.loc[df['IS_ANOMALY'] == True].copy()
anomalies['TS'] = pd.to_datetime(anomalies['TS']).dt.date
# print(anomalies.tolist())

# Original Anomalies injected by me
out = []
last_year_start_index = (len(df_data) // 3) * 2
indexes = [
        last_year_start_index + 1000, 
        last_year_start_index + 3000, 
        last_year_start_index + 5000, 
        last_year_start_index + 7000, 
        last_year_start_index + 9000
]
for idx in indexes:
    out.append(df_data.loc[idx])
# print(len(out))
out_df = pd.DataFrame(out)

# ensure both are datetime.date
out_df['as_of_date'] = pd.to_datetime(out_df['as_of_date']).dt.date
anomalies['TS'] = pd.to_datetime(anomalies['TS']).dt.date


# Now check if these records are present in the anomalies as well
matched = anomalies.merge(
    out_df,
    right_on=['as_of_date', 'value'],
    left_on=['TS', 'Y'],
    how='inner'
)

matched.head()


,SERIES,TS,Y,FORECAST,LOWER_BOUND,UPPER_BOUND,IS_ANOMALY,PERCENTILE,DISTANCE,as_of_date,key_1,key_2,key_3,value


In [54]:
missed = out_df.merge(
    anomalies,
    right_on=['TS', 'Y'],
    left_on=['as_of_date', 'value'],
    how='left',
    indicator=True
).query("_merge == 'left_only'")

missed.head()


,as_of_date,key_1,key_2,key_3,value,SERIES,TS,Y,FORECAST,LOWER_BOUND,UPPER_BOUND,IS_ANOMALY,PERCENTILE,DISTANCE,_merge
0,2026-02-22,key_1_b,key_2_a,key_3_b,3544.997677,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,2026-05-07,key_1_b,key_2_b,key_3_a,3545.001747,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,2026-07-20,key_1_b,key_2_b,key_3_c,3544.998829,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,2026-10-02,key_1_b,key_2_c,key_3_b,3544.997603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,2026-12-15,key_1_c,key_2_a,key_3_a,3544.996780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [56]:
given_date = pd.to_datetime('2026-02-22').date()
records = anomalies[anomalies['TS'] == given_date]
print(records)

                                             SERIES          TS            Y  \
1496  [\n  "key_1_b",\n  "key_2_a",\n  "key_3_b"\n]  2026-02-22  3544.997677   

         FORECAST  LOWER_BOUND  UPPER_BOUND  IS_ANOMALY  PERCENTILE  \
1496  2344.998887  2344.993199  2345.004574        True         1.0   

           DISTANCE  
1496  543475.906733  
